# GSE5406: Stage 1 sample manifest and Stage 2 QC

This notebook documents the GEO processed-matrix retrieval and validation, sample manifest, and Stage 2 quality-control review for GSE5406. It does not fit a limma model or process GSE42955.

Group labels come from GEO `!Sample_characteristics_ch1`; original titles are retained from `!Sample_title`. Ischemic cardiomyopathy samples are recorded as excluded with the label-based reason. Stage 2 QC uses the 86 DCM and 16 non-failing controls; no sample is excluded based on QC flags.

In [1]:
# Locate the project root and retrieve the processed GEO series matrix if needed.
root_candidates <- c(".", "..", "../..")
root_candidates <- root_candidates[file.exists(file.path(root_candidates, "plan/analysis_plan.md"))]
if (length(root_candidates) == 0L) stop("Could not locate project root from the notebook working directory.")
project_root <- normalizePath(root_candidates[[1]], winslash = "/")
raw_dir <- file.path(project_root, "data", "raw")
dir.create(raw_dir, recursive = TRUE, showWarnings = FALSE)
matrix_file <- file.path(raw_dir, "GSE5406_series_matrix.txt.gz")
source_url <- "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE5nnn/GSE5406/matrix/GSE5406_series_matrix.txt.gz"
if (!file.exists(matrix_file)) {
  download.file(source_url, destfile = matrix_file, mode = "wb", method = "libcurl", quiet = FALSE)
}
stopifnot(file.exists(matrix_file), file.info(matrix_file)$size > 0)
cat("Source:", source_url, "\n")
cat("Saved to:", matrix_file, "\n")
cat("Compressed file size:", file.info(matrix_file)$size, "bytes\n")
cat("MD5:", unname(tools::md5sum(matrix_file)), "\n")

Source: https://ftp.ncbi.nlm.nih.gov/geo/series/GSE5nnn/GSE5406/matrix/GSE5406_series_matrix.txt.gz 


Saved to: /home/user/biom5350/biom5350-dcm-consensus/data/raw/GSE5406_series_matrix.txt.gz 


Compressed file size: 23607082 bytes


MD5: 761e0771018f2c6de701e6989dc9b55e 


## Read GEO sample metadata and create the manifest

The GEO series matrix records the sample title, accession, characteristics, platform, and processing annotation before its expression table. The characteristics field is the source used to assign DCM, non-failing control, or ICM. Any unrecognized label is kept visible as `review required` rather than silently assigned.

In [2]:
# Read sample metadata lines, then parse the matrix header and processed values.
con <- gzfile(matrix_file, open = "rt")
sample_lines <- character()
repeat {
  line <- readLines(con, n = 1L, warn = FALSE)
  if (length(line) == 0L) stop("GEO series matrix ended before its sample table.")
  if (identical(line, "!series_matrix_table_begin")) break
  if (startsWith(line, "!Sample_")) sample_lines <- c(sample_lines, line)
}
parse_sample_line <- function(line) {
  fields <- read.delim(text = line, sep = "\t", header = FALSE, quote = "\"",
                       comment.char = "", check.names = FALSE, stringsAsFactors = FALSE)
  list(
    key = sub("^!Sample_", "", as.character(fields[1L, 1L])),
    values = as.character(unlist(fields[1L, -1L, drop = FALSE], use.names = FALSE))
  )
}
records <- lapply(sample_lines, parse_sample_line)
metadata_fields <- lapply(records, function(x) x[["values"]])
names(metadata_fields) <- vapply(records, function(x) x[["key"]], character(1))
if (anyDuplicated(names(metadata_fields))) stop("Repeated GEO sample metadata fields require review.")
if (length(unique(lengths(metadata_fields))) != 1L) stop("GEO sample metadata fields have inconsistent lengths.")
sample_metadata <- tibble::as_tibble(metadata_fields, .name_repair = "minimal")

# Read the processed expression table only for platform, sample-column, and scale validation.
if (!"data_row_count" %in% names(sample_metadata)) stop("GEO metadata lacks !Sample_data_row_count.")
row_counts <- unique(as.integer(sample_metadata$data_row_count))
if (length(row_counts) != 1L || is.na(row_counts)) stop("GEO matrix row count is inconsistent.")
expression_table <- read.delim(con, sep = "\t", header = TRUE, nrows = row_counts,
                               comment.char = "", check.names = FALSE, stringsAsFactors = FALSE)
close(con)

# Assign groups from the explicit GEO characteristics field and retain both source labels.
characteristics <- sample_metadata$characteristics_ch1 |>
  stringr::str_trim() |>
  stringr::str_to_lower()
assigned_group <- dplyr::case_when(
  stringr::str_detect(characteristics, stringr::fixed("normally functioning myocardium from unused donor heart")) ~ "non-failing control",
  stringr::str_detect(characteristics, stringr::fixed("idiopathic dilated cardiomyopathy")) ~ "DCM",
  stringr::str_detect(characteristics, stringr::fixed("ischemic cardiomyopathy")) ~ "ischemic cardiomyopathy",
  TRUE ~ "Unclassified"
)
inclusion_status <- dplyr::case_when(
  assigned_group %in% c("DCM", "non-failing control") ~ "included",
  assigned_group == "Unclassified" ~ "review required",
  TRUE ~ "excluded"
)
decision_reason <- dplyr::case_when(
  assigned_group == "DCM" ~ "Included: GEO characteristics explicitly label idiopathic dilated cardiomyopathy",
  assigned_group == "non-failing control" ~ "Included: GEO characteristics explicitly label normally functioning myocardium from an unused donor heart",
  assigned_group == "ischemic cardiomyopathy" ~ "Excluded: GEO characteristics explicitly label ischemic cardiomyopathy",
  TRUE ~ "Review required: GEO characteristics did not match an approved group label"
)
sample_manifest <- tibble::tibble(
  gsm_id = sample_metadata$geo_accession,
  original_title = sample_metadata$title,
  original_characteristics = sample_metadata$characteristics_ch1,
  assigned_group = assigned_group,
  inclusion_status = inclusion_status,
  decision_reason = decision_reason,
  label_source = "GEO GSE5406 series matrix: !Sample_characteristics_ch1; title from !Sample_title"
)
if (anyDuplicated(sample_manifest$gsm_id)) stop("Duplicate GSM accessions found.")
manifest_path <- file.path(project_root, "results", "tables", "GSE5406_sample_info.csv")
dir.create(dirname(manifest_path), recursive = TRUE, showWarnings = FALSE)
write.csv(sample_manifest, manifest_path, row.names = FALSE, na = "")
cat("Manifest saved:", manifest_path, "\n")
cat("Samples in manifest:", nrow(sample_manifest), "\n")


Manifest saved: /home/user/biom5350/biom5350-dcm-consensus/results/tables/GSE5406_sample_info.csv 


Samples in manifest: 210 


## Stage 1 checks

The expected counts are 86 DCM, 16 non-failing controls, and 108 ICM exclusions. Check the actual label counts, platform, metadata-to-matrix sample mapping, feature-row count, and the GEO processing annotation. RMA produces log2-scale expression summaries; the observed value range is recorded as a consistency check. This is matrix validation only, not QC.

In [3]:
# Compare observed groups and matrix metadata with the approved plan.
expected_counts <- c(DCM = 86L, `non-failing control` = 16L, `ischemic cardiomyopathy` = 108L, Unclassified = 0L)
expected_group_counts <- tibble::tibble(
  assigned_group = names(expected_counts),
  expected = as.integer(expected_counts)
)
observed_counts <- dplyr::count(sample_manifest, assigned_group, name = "observed")
count_check <- expected_group_counts |>
  dplyr::left_join(observed_counts, by = "assigned_group") |>
  dplyr::mutate(
    observed = tidyr::replace_na(observed, 0L),
    matches_expected = observed == expected
  )
print(count_check, n = Inf)

# Confirm the GEO platform and processing annotation before interpreting expression values.
platforms <- unique(sample_metadata$platform_id)
processing <- unique(sample_metadata$data_processing)
matrix_sample_ids <- names(expression_table)[-1L]
metadata_sample_ids <- sample_manifest$gsm_id
expression_values <- as.matrix(expression_table[, -1L, drop = FALSE])
suppressWarnings(storage.mode(expression_values) <- "numeric")
value_range <- range(expression_values, na.rm = TRUE)

# Check sample identity, expected feature count, and whether the observed range fits log2 RMA values.
validation <- tibble::tibble(
  check = c("210 metadata records", "unique GSM IDs", "expected group counts", "GPL96 platform",
            "RMA processing annotation", "matrix columns match metadata GSM IDs",
            "feature row count matches GEO metadata", "numeric range compatible with log2 RMA values"),
  observed = c(
    as.character(nrow(sample_metadata)),
    as.character(length(unique(metadata_sample_ids))),
    paste(count_check$assigned_group, count_check$observed, sep = "=", collapse = "; "),
    paste(platforms, collapse = "; "),
    paste(processing, collapse = "; "),
    as.character(setequal(matrix_sample_ids, metadata_sample_ids) && length(matrix_sample_ids) == length(metadata_sample_ids)),
    paste(nrow(expression_table), row_counts, sep = " / "),
    paste(signif(value_range, 5), collapse = " to ")
  ),
  status = c(
    ifelse(nrow(sample_metadata) == 210L, "PASS", "MISMATCH"),
    ifelse(length(unique(metadata_sample_ids)) == 210L, "PASS", "MISMATCH"),
    ifelse(all(count_check$matches_expected), "PASS", "MISMATCH"),
    ifelse(length(platforms) == 1L && identical(platforms, "GPL96"), "PASS", "MISMATCH"),
    ifelse(length(processing) == 1L && grepl("RMA", processing, ignore.case = TRUE), "PASS", "MISMATCH"),
    ifelse(setequal(matrix_sample_ids, metadata_sample_ids) && length(matrix_sample_ids) == length(metadata_sample_ids), "PASS", "MISMATCH"),
    ifelse(nrow(expression_table) == row_counts, "PASS", "MISMATCH"),
    ifelse(all(is.finite(value_range)) && value_range[1] > 0 && value_range[2] < 30, "CONSISTENT", "REVIEW")
  )
)
print(validation, n = Inf)

# Save the validation summary and report the matrix dimensions without fitting a model.
validation_path <- file.path(project_root, "results", "tables", "GSE5406_stage1_validation.csv")
write.csv(validation, validation_path, row.names = FALSE)
cat("Validation summary saved:", validation_path, "\n")
cat("Expression matrix dimensions (features x samples):", nrow(expression_table), "x", length(matrix_sample_ids), "\n")
cat("No QC plots or differential-expression models were run in this notebook stage.\n")


# A tibble: 4 × 4
  assigned_group          expected observed matches_expected
  <chr>                      <int>    <int> <lgl>           
1 DCM                           86       86 TRUE            
2 non-failing control           16       16 TRUE            
3 ischemic cardiomyopathy      108      108 TRUE            
4 Unclassified                   0        0 TRUE            


# A tibble: 8 × 3
  check                                         observed                  status
  <chr>                                         <chr>                     <chr> 
1 210 metadata records                          210                       PASS  
2 unique GSM IDs                                210                       PASS  
3 expected group counts                         DCM=86; non-failing cont… PASS  
4 GPL96 platform                                GPL96                     PASS  
5 RMA processing annotation                     Data were normalized usi… PASS  
6 matrix columns match metadata GSM IDs         TRUE                      PASS  
7 feature row count matches GEO metadata        22283 / 22283             PASS  
8 numeric range compatible with log2 RMA values 2.9939 to 13.399          CONSI…


Validation summary saved: /home/user/biom5350/biom5350-dcm-consensus/results/tables/GSE5406_stage1_validation.csv 


Expression matrix dimensions (features x samples): 22283 x 210 


No QC plots or differential-expression models were run in this notebook stage.


## Reproducibility record

Record R and package versions for this Stage 1 run. The notebook uses base R for metadata parsing and CSV export; no GEOquery installation is required.

In [4]:
# Record the R environment used to create the Stage 1 outputs.
sessionInfo()

R version 4.5.2 (2025-10-31)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 26.04 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/blas/libblas.so.3.12.1 
LAPACK: /usr/lib/x86_64-linux-gnu/lapack/liblapack.so.3.12.1;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

loaded via a namespace (and not attached):
 [1] crayon_1.5.3      vctrs_0.7.3       cli_3.6.6         rlang_1.3.0      
 [5] stringi_1.8.9     purrr_1.2.2       generics_0.1.4    textshaping_1.0.5
 [9] jsonlite_2.0.0    glue_1.8.1        htmltools_0.5.9   IRdisplay_1.1    
[13] IRkern

## Stage 2: GSE5406 sample-level quality control

QC uses the 102 explicitly labeled DCM and non-failing control samples; the 108 ischemic samples remain excluded by the approved sample-selection rule. No included sample is removed based on QC. Before PCA, MDS, and clustering, report missing and non-finite values, retain features finite across every included sample, and omit exact zero-variance features. No values are imputed. Save an expression-distribution plot, PCA, classical MDS, correlation-distance sample dendrogram, and a per-sample QC summary. Potential outlier flags are review prompts only.


In [5]:
# Prepare included samples in manifest order and check matrix values before QC.
included_rows <- which(sample_manifest$inclusion_status == "included")
qc_sample_ids <- sample_manifest$gsm_id[included_rows]
qc_columns <- match(qc_sample_ids, colnames(expression_values))
if (anyNA(qc_columns) || anyDuplicated(qc_sample_ids)) {
  stop("Included GSM IDs do not map uniquely to the processed matrix columns.")
}
qc_expr_all <- expression_values[, qc_columns, drop = FALSE]
qc_groups <- factor(sample_manifest$assigned_group[included_rows],
                     levels = c("non-failing control", "DCM"))
if (anyNA(qc_groups)) stop("Unexpected group label among included samples.")

# Report the processed matrix scale range and all missing/non-finite values.
full_finite_values <- expression_values[is.finite(expression_values)]
included_finite_values <- qc_expr_all[is.finite(qc_expr_all)]
full_value_range <- range(full_finite_values)
included_value_range <- range(included_finite_values)
missing_cells <- sum(is.na(qc_expr_all) & !is.nan(qc_expr_all))
nan_cells <- sum(is.nan(qc_expr_all))
infinite_cells <- sum(is.infinite(qc_expr_all))
nonfinite_feature <- rowSums(!is.finite(qc_expr_all)) > 0L
complete_feature <- rowSums(is.finite(qc_expr_all)) == ncol(qc_expr_all)
complete_expr <- qc_expr_all[complete_feature, , drop = FALSE]

# Exclude incomplete and exact zero-variance features from PCA, MDS, and clustering only.
feature_sd <- apply(complete_expr, 1L, stats::sd)
zero_variance_feature <- !is.finite(feature_sd) | feature_sd == 0
qc_expr <- complete_expr[!zero_variance_feature, , drop = FALSE]
if (nrow(qc_expr) < 2L) {
  stop("Too few finite, variable features for QC; pause before imputation or alternatives.")
}
cat("Included samples:", ncol(qc_expr_all), "\n")
cat("GEO processing annotation: RMA; RMA expression summaries are on the log2 scale.\n")
cat("Finite value range across all 210 matrix samples:", signif(full_value_range, 6), "\n")
cat("Finite value range across included samples:", signif(included_value_range, 6), "\n")
cat("Included-sample NA cells (excluding NaN):", missing_cells, "\n")
cat("Included-sample NaN cells:", nan_cells, "\n")
cat("Included-sample positive/negative Inf cells:", infinite_cells, "\n")
cat("Features with any missing or non-finite included value:", sum(nonfinite_feature), "\n")
cat("Features finite across every included sample:", sum(complete_feature), "\n")
cat("Exact zero-variance features among complete features:", sum(zero_variance_feature), "\n")
cat("Features used for PCA, MDS, and clustering:", nrow(qc_expr), "\n")

# Summarize each sample's expression distribution without imputing values.
finite_values_by_sample <- lapply(seq_len(ncol(qc_expr_all)), function(j) {
  x <- qc_expr_all[, j]
  x[is.finite(x)]
})
names(finite_values_by_sample) <- colnames(qc_expr_all)
qc_metrics <- tibble::tibble(
  gsm_id = colnames(qc_expr_all),
  assigned_group = as.character(qc_groups),
  nonfinite_features = colSums(!is.finite(qc_expr_all)),
  median_expression = apply(qc_expr_all, 2L, stats::median, na.rm = TRUE),
  mean_expression = apply(qc_expr_all, 2L, mean, na.rm = TRUE),
  sd_expression = apply(qc_expr_all, 2L, stats::sd, na.rm = TRUE),
  q1_expression = apply(qc_expr_all, 2L, stats::quantile, probs = 0.25, na.rm = TRUE),
  q3_expression = apply(qc_expr_all, 2L, stats::quantile, probs = 0.75, na.rm = TRUE)
)

# Use finite, non-constant features for PCA, classical MDS, and correlation clustering.
pca_fit <- stats::prcomp(t(qc_expr), center = TRUE, scale. = FALSE)
pca_variance <- pca_fit$sdev^2 / sum(pca_fit$sdev^2)
pca_scores <- pca_fit$x[, 1:2, drop = FALSE]
mds_fit <- stats::cmdscale(stats::dist(t(qc_expr)), k = 2L, eig = TRUE, add = TRUE)
mds_scores <- mds_fit$points
sample_correlation <- stats::cor(qc_expr, method = "pearson")
correlation_distance <- stats::as.dist(pmax(1 - sample_correlation, 0))
cluster_fit <- stats::hclust(correlation_distance, method = "average")

# Flag review candidates with extreme within-group QC metrics; never remove them.
robust_z <- function(x) {
  center <- stats::median(x, na.rm = TRUE)
  spread <- stats::mad(x, constant = 1.4826, na.rm = TRUE)
  if (!is.finite(spread) || spread == 0) return(rep(NA_real_, length(x)))
  (x - center) / spread
}
qc_metrics$median_robust_z <- ave(qc_metrics$median_expression, qc_metrics$assigned_group,
                                  FUN = robust_z)
qc_metrics$pca1_robust_z <- ave(pca_scores[, 1L], qc_metrics$assigned_group, FUN = robust_z)
qc_metrics$pca2_robust_z <- ave(pca_scores[, 2L], qc_metrics$assigned_group, FUN = robust_z)
within_group_mean_cor <- vapply(seq_len(ncol(sample_correlation)), function(i) {
  peers <- which(qc_groups == qc_groups[i] & seq_along(qc_groups) != i)
  if (length(peers) == 0L) NA_real_ else mean(sample_correlation[i, peers])
}, numeric(1))
qc_metrics$within_group_mean_correlation <- within_group_mean_cor
qc_metrics$within_group_correlation_robust_z <- ave(
  within_group_mean_cor, qc_metrics$assigned_group, FUN = robust_z
)
qc_metrics$potential_outlier <- with(qc_metrics,
  abs(median_robust_z) > 3 | abs(pca1_robust_z) > 3 | abs(pca2_robust_z) > 3 |
    within_group_correlation_robust_z < -3
)
qc_metrics$potential_outlier[is.na(qc_metrics$potential_outlier)] <- FALSE
qc_metrics$flag_reason <- vapply(seq_len(nrow(qc_metrics)), function(i) {
  reasons <- character()
  if (is.finite(qc_metrics$median_robust_z[i]) && abs(qc_metrics$median_robust_z[i]) > 3)
    reasons <- c(reasons, "sample median >3 group-wise robust SD")
  if (is.finite(qc_metrics$pca1_robust_z[i]) && abs(qc_metrics$pca1_robust_z[i]) > 3)
    reasons <- c(reasons, "PC1 >3 group-wise robust SD")
  if (is.finite(qc_metrics$pca2_robust_z[i]) && abs(qc_metrics$pca2_robust_z[i]) > 3)
    reasons <- c(reasons, "PC2 >3 group-wise robust SD")
  if (is.finite(qc_metrics$within_group_correlation_robust_z[i]) &&
      qc_metrics$within_group_correlation_robust_z[i] < -3)
    reasons <- c(reasons, "low within-group mean correlation")
  if (length(reasons) == 0L) "" else paste(reasons, collapse = "; ")
}, character(1))

# Save the per-sample summary so plotted samples can be traced to GSM IDs.
figure_dir <- file.path(project_root, "results", "figures")
table_dir <- file.path(project_root, "results", "tables")
dir.create(figure_dir, recursive = TRUE, showWarnings = FALSE)
write.csv(qc_metrics, file.path(table_dir, "GSE5406_QC_sample_metrics.csv"), row.names = FALSE)

# Reshape finite expression values for tidy, comparable per-sample boxplots.
plot_order <- order(qc_groups, qc_metrics$gsm_id)
plot_colors <- c("non-failing control" = "#2B8CBE", "DCM" = "#D7301F")
plot_vector_colors <- unname(plot_colors[qc_metrics$assigned_group[plot_order]])
qc_distribution_data <- as.data.frame(qc_expr_all, check.names = FALSE) |>
  tibble::as_tibble(.name_repair = "minimal") |>
  tidyr::pivot_longer(
    cols = dplyr::everything(),
    names_to = "gsm_id",
    values_to = "expression"
  ) |>
  dplyr::filter(is.finite(expression)) |>
  dplyr::left_join(dplyr::select(qc_metrics, gsm_id, assigned_group), by = "gsm_id") |>
  dplyr::mutate(gsm_id = factor(gsm_id, levels = qc_metrics$gsm_id[plot_order]))
expression_plot <- ggplot2::ggplot(
  qc_distribution_data,
  ggplot2::aes(x = gsm_id, y = expression, fill = assigned_group)
) +
  ggplot2::geom_boxplot(outlier.shape = NA, linewidth = 0.25) +
  ggplot2::geom_vline(
    xintercept = sum(qc_groups == "non-failing control") + 0.5,
    linetype = 2, color = "grey35"
  ) +
  ggplot2::scale_fill_manual(values = plot_colors) +
  ggplot2::labs(
    title = "GSE5406 per-sample expression distributions",
    y = "Processed log2 expression",
    x = "Included sample (GSM)",
    fill = NULL
  ) +
  ggplot2::theme_bw() +
  ggplot2::theme(
    axis.text.x = ggplot2::element_text(angle = 90, hjust = 1, vjust = 0.5, size = 5)
  )
ggplot2::ggsave(
  file.path(figure_dir, "GSE5406_expression_distributions.png"),
  plot = expression_plot, width = 26.67, height = 10.56, dpi = 180
)

# Plot the first two centered, unscaled principal components and label flagged samples.
pca_plot_data <- tibble::tibble(
  gsm_id = qc_metrics$gsm_id,
  PC1 = unname(pca_scores[, 1L]),
  PC2 = unname(pca_scores[, 2L]),
  assigned_group = qc_metrics$assigned_group,
  potential_outlier = qc_metrics$potential_outlier
)
pca_plot <- ggplot2::ggplot(
  pca_plot_data,
  ggplot2::aes(x = PC1, y = PC2, color = assigned_group)
) +
  ggplot2::geom_point(size = 2) +
  ggplot2::scale_color_manual(values = plot_colors) +
  ggplot2::labs(
    x = sprintf("PC1 (%.1f%%)", 100 * pca_variance[1L]),
    y = sprintf("PC2 (%.1f%%)", 100 * pca_variance[2L]),
    title = "GSE5406 PCA: included samples",
    color = NULL
  ) +
  ggplot2::theme_bw()
if (any(pca_plot_data$potential_outlier)) {
  pca_plot <- pca_plot +
    ggplot2::geom_text(
      data = dplyr::filter(pca_plot_data, potential_outlier),
      ggplot2::aes(label = gsm_id), vjust = -0.6, size = 2.5, show.legend = FALSE
    )
}
ggplot2::ggsave(
  file.path(figure_dir, "GSE5406_PCA.png"),
  plot = pca_plot, width = 13.33, height = 10.56, dpi = 180
)

# Plot classical MDS of Euclidean sample distances across QC-eligible features.
mds_plot_data <- tibble::tibble(
  gsm_id = qc_metrics$gsm_id,
  MDS1 = unname(mds_scores[, 1L]),
  MDS2 = unname(mds_scores[, 2L]),
  assigned_group = qc_metrics$assigned_group,
  potential_outlier = qc_metrics$potential_outlier
)
mds_plot <- ggplot2::ggplot(
  mds_plot_data,
  ggplot2::aes(x = MDS1, y = MDS2, color = assigned_group)
) +
  ggplot2::geom_point(size = 2) +
  ggplot2::scale_color_manual(values = plot_colors) +
  ggplot2::labs(
    x = "Classical MDS dimension 1",
    y = "Classical MDS dimension 2",
    title = "GSE5406 MDS: Euclidean distance",
    color = NULL
  ) +
  ggplot2::theme_bw()
if (any(mds_plot_data$potential_outlier)) {
  mds_plot <- mds_plot +
    ggplot2::geom_text(
      data = dplyr::filter(mds_plot_data, potential_outlier),
      ggplot2::aes(label = gsm_id), vjust = -0.6, size = 2.5, show.legend = FALSE
    )
}
ggplot2::ggsave(
  file.path(figure_dir, "GSE5406_MDS.png"),
  plot = mds_plot, width = 13.33, height = 10.56, dpi = 180
)

# Keep the specialized hierarchical-clustering dendrogram in base graphics.
png(file.path(figure_dir, "GSE5406_sample_clustering.png"),
    width = 4800, height = 2200, res = 180)
par(mar = c(12, 4, 4, 1))
plot(cluster_fit, hang = -1, cex = 0.38, xlab = "Sample GSM",
     main = "GSE5406 sample clustering (1 - Pearson correlation)")
invisible(dev.off())

# Summarize data-quality checks and any samples flagged for review.
cat("QC-eligible feature count:", nrow(qc_expr), "\n")
cat("QC figures saved under:", figure_dir, "\n")
cat("Sample metrics saved to: results/tables/GSE5406_QC_sample_metrics.csv\n")
flagged_samples <- qc_metrics |>
  dplyr::filter(potential_outlier) |>
  dplyr::select(gsm_id, assigned_group, flag_reason)
if (nrow(flagged_samples) == 0L) {
  cat("Potential sample outlier flags: none by the stated exploratory thresholds.\n")
} else {
  cat("Potential sample outlier flags (review only; none excluded):\n")
  print(flagged_samples, n = Inf)
}
cat("No samples were excluded, and no limma model was fitted.\n")


Included samples: 102 


GEO processing annotation: RMA; RMA expression summaries are on the log2 scale.


Finite value range across all 210 matrix samples: 2.99395 13.3988 


Finite value range across included samples: 2.99395 13.3988 


Included-sample NA cells (excluding NaN): 0 


Included-sample NaN cells: 0 


... (6 outputs left out)

QC figures saved under: /home/user/biom5350/biom5350-dcm-consensus/results/figures 


Sample metrics saved to: results/tables/GSE5406_QC_sample_metrics.csv


Potential sample outlier flags (review only; none excluded):
# A tibble: 17 × 3
   gsm_id    assigned_group      flag_reason                                    
   <chr>     <chr>               <chr>                                          
 1 GSM123533 DCM                 PC2 >3 group-wise robust SD                    
 2 GSM123601 DCM                 low within-group mean correlation              
 3 GSM123610 DCM                 PC2 >3 group-wise robust SD; low within-group …
 4 GSM123649 non-failing control low within-group mean correlation              
 5 GSM123650 DCM                 PC1 >3 group-wise robust SD; low within-group …
 6 GSM123651 DCM                 PC1 >3 group-wise robust SD; low within-group …
 7 GSM123652 DCM                 PC1 >3 group-wise robust SD; low within-group …
 8 GSM123653 DCM                 PC1 >3 group-wise robust SD                    
 9 GSM123654 DCM                 PC1 >3 group-wise robust SD; low within-group …
10 GSM123657 DCM             

No samples were excluded, and no limma model was fitted.


## Stage 2 QC findings

The GEO record identifies the processed matrix as RMA normalized; RMA summaries are on the log2 scale. The observed finite range is 2.99395 to 13.3988 across all 210 matrix samples and is the same across the 102 included samples. Among included samples there were 0 NA, 0 NaN, and 0 infinite cells. All 22,283 features were finite across all included samples, and none had zero variance, so all 22,283 were used for PCA, MDS, and clustering. No imputation was needed.

Per-sample expression distributions have closely aligned medians and quartiles, with no obvious global shift for an individual sample. PCA explains 15.1% and 12.3% of variance on its first two components. Classical MDS uses the same Euclidean sample distances, so its configuration mirrors PCA. The plots show a separated subset of DCM samples and the correlation dendrogram has several longer branches; these patterns may reflect biological heterogeneity or technical structure, which cannot be distinguished from the available covariates.

The exploratory group-wise robust-score rule flagged 17 samples for review: 16 DCM samples (GSM123533, GSM123601, GSM123610, GSM123650, GSM123651, GSM123652, GSM123653, GSM123654, GSM123657, GSM123661, GSM123663, GSM123667, GSM123687, GSM123711, GSM123719, GSM123720) and one non-failing control (GSM123649). Flags arise from extreme PC scores and/or low within-group mean correlation; they are review prompts, not evidence of sample failure. No sample was excluded. The sample-level metrics CSV records the flag reasons.


## Stage 3 differential expression: GSE5406

Retain all 102 approved samples. Before testing, identify GPL96 platform control probe sets from the platform annotation and map biological probes to gene symbols. Fit limma with disease status only, using non-failing controls as the reference and the DCM-minus-control contrast. Save full probe-level and one-probe-per-symbol results, plus significant subsets. Ambiguous or unmapped probes remain in probe-level tables but are omitted from gene-symbol selections.

In [6]:
# Inspect GPL96 identifiers and use a writable user library for the platform map.
feature_ids <- as.character(expression_table[[1L]])
control_probe <- grepl("^AFFX", feature_ids, ignore.case = TRUE)
cat("Processed matrix dimensions (features x samples):", nrow(expression_values), "x", ncol(expression_values), "\n")
cat("Unique feature identifiers:", length(unique(feature_ids)), "\n")
cat("GPL96 AFFX control probes:", sum(control_probe), "\n")
cat("Biological probes to test:", sum(!control_probe), "\n")
user_package_lib <- file.path(path.expand("~"), "R", "library")
dir.create(user_package_lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(user_package_lib, .libPaths()))
if (!requireNamespace("hgu133a.db", quietly = TRUE)) {
  options(repos = BiocManager::repositories())
  BiocManager::install("hgu133a.db", lib = user_package_lib,
                       ask = FALSE, update = FALSE, quiet = TRUE)
}
if (!requireNamespace("hgu133a.db", quietly = TRUE)) {
  stop("The GPL96 annotation package could not be installed; pause before modeling.")
}
cat("GPL96 annotation source: Bioconductor hgu133a.db",
    as.character(utils::packageVersion("hgu133a.db")), "\n")
cat("limma:", as.character(utils::packageVersion("limma")), "\n")


Processed matrix dimensions (features x samples): 22283 x 210 


Unique feature identifiers: 22283 


GPL96 AFFX control probes: 68 


Biological probes to test: 22215 


GPL96 annotation source: Bioconductor hgu133a.db 3.13.0 


limma: 3.66.0 


In [7]:
# Add the writable user library, then load limma and GPL96 probe annotation.
user_package_lib <- file.path(path.expand("~"), "R", "library")
.libPaths(c(user_package_lib, .libPaths()))
suppressPackageStartupMessages(library(limma))
suppressPackageStartupMessages(library(AnnotationDbi))
suppressPackageStartupMessages(library(hgu133a.db))

# Keep all 102 approved samples and remove only GPL96 AFFX control probes.
feature_ids <- as.character(expression_table[[1L]])
control_probe <- grepl("^AFFX", feature_ids, ignore.case = TRUE)
biological_probe_ids <- feature_ids[!control_probe]
if (anyDuplicated(feature_ids)) stop("Feature IDs are not unique; pause before modeling.")
if (sum(control_probe) != 68L) stop("The observed GPL96 control-probe count changed; review annotation.")
if (length(biological_probe_ids) != 22215L) stop("Unexpected number of biological probes.")
included_rows <- which(sample_manifest$inclusion_status == "included")
included_ids <- sample_manifest$gsm_id[included_rows]
de_columns <- match(included_ids, colnames(expression_values))
if (anyNA(de_columns) || length(de_columns) != 102L) stop("Included GSMs do not map to 102 matrix columns.")
expression_test <- expression_values[!control_probe, de_columns, drop = FALSE]
rownames(expression_test) <- biological_probe_ids
if (!all(is.finite(expression_test))) stop("Non-finite expression values found; pause before limma.")

# Map each biological probe with the Bioconductor GPL96 annotation package.
annotation_rows <- AnnotationDbi::select(
  hgu133a.db, keys = biological_probe_ids,
  keytype = "PROBEID", columns = c("SYMBOL", "GENENAME", "ENTREZID")
)
annotation_rows <- annotation_rows[!is.na(annotation_rows$PROBEID), , drop = FALSE]
annotation_split <- split(annotation_rows, as.character(annotation_rows$PROBEID))
get_annotation_values <- function(probe_id, field) {
  rows <- annotation_split[[probe_id]]
  if (is.null(rows)) return(character())
  values <- unique(as.character(rows[[field]]))
  sort(values[!is.na(values) & nzchar(values)])
}
annotation_map <- tibble::tibble(
  probe_id = biological_probe_ids,
  all_gene_symbols = vapply(biological_probe_ids, function(id) {
    values <- get_annotation_values(id, "SYMBOL")
    if (length(values)) paste(values, collapse = ";") else NA_character_
  }, character(1)),
  gene_name = vapply(biological_probe_ids, function(id) {
    values <- get_annotation_values(id, "GENENAME")
    if (length(values)) paste(values, collapse = ";") else NA_character_
  }, character(1)),
  entrez_id = vapply(biological_probe_ids, function(id) {
    values <- get_annotation_values(id, "ENTREZID")
    if (length(values)) paste(values, collapse = ";") else NA_character_
  }, character(1))
)
symbol_counts <- vapply(biological_probe_ids, function(id) {
  length(get_annotation_values(id, "SYMBOL"))
}, integer(1))
annotation_map$annotation_status <- ifelse(
  symbol_counts == 0L, "unmapped",
  ifelse(symbol_counts > 1L, "ambiguous", "mapped")
)
annotation_map$gene_symbol <- vapply(biological_probe_ids, function(id) {
  values <- get_annotation_values(id, "SYMBOL")
  if (length(values) == 1L) values else NA_character_
}, character(1))

# Set non-failing controls as the reference; groupDCM is DCM minus control.
group <- stats::relevel(
  factor(sample_manifest$assigned_group[included_rows],
         levels = c("non-failing control", "DCM")),
  ref = "non-failing control"
)
if (anyNA(group) || sum(group == "DCM") != 86L || sum(group == "non-failing control") != 16L) {
  stop("Unexpected analysis group counts; pause before limma.")
}
design <- stats::model.matrix(~ group)
if (!"groupDCM" %in% colnames(design)) stop("Expected DCM-minus-control coefficient is absent.")
fit <- limma::eBayes(limma::lmFit(expression_test, design))
limma_table <- limma::topTable(
  fit, coef = "groupDCM", number = Inf,
  adjust.method = "BH", sort.by = "none"
)
limma_table$probe_id <- rownames(limma_table)
limma_table <- limma_table[match(biological_probe_ids, limma_table$probe_id), , drop = FALSE]
if (anyNA(limma_table$probe_id)) stop("Some biological probes are missing from limma output.")

# Add group means and preserve every tested biological probe, including unmapped probes.
mean_control <- rowMeans(expression_test[, group == "non-failing control", drop = FALSE])
mean_dcm <- rowMeans(expression_test[, group == "DCM", drop = FALSE])
probe_results <- tibble::tibble(
  probe_id = limma_table$probe_id,
  gene_symbol = annotation_map$gene_symbol,
  all_gene_symbols = annotation_map$all_gene_symbols,
  gene_name = annotation_map$gene_name,
  entrez_id = annotation_map$entrez_id,
  annotation_status = annotation_map$annotation_status,
  mean_control = mean_control,
  mean_DCM = mean_dcm,
  logFC = limma_table$logFC,
  AveExpr = limma_table$AveExpr,
  t = limma_table$t,
  P.Value = limma_table$P.Value,
  adj.P.Val = limma_table$adj.P.Val,
  B = limma_table$B
)
if (max(abs((probe_results$mean_DCM - probe_results$mean_control) - probe_results$logFC)) > 1e-8) {
  stop("Reported logFC does not match DCM minus control group means.")
}
probe_results <- dplyr::arrange(
  probe_results, adj.P.Val, dplyr::desc(abs(logFC)), probe_id, .locale = "C"
)

# Select one uniquely mapped probe per symbol using the approved tie-breaking rule.
mapped_probes <- probe_results |>
  dplyr::filter(annotation_status == "mapped") |>
  dplyr::arrange(
    gene_symbol, adj.P.Val, dplyr::desc(abs(logFC)), probe_id, .locale = "C"
  )
gene_symbol_results <- mapped_probes |>
  dplyr::distinct(gene_symbol, .keep_all = TRUE) |>
  dplyr::arrange(adj.P.Val, gene_symbol, .locale = "C")

# Create the four requested result tables and a compact count summary.
significant_probe_results <- dplyr::filter(probe_results, adj.P.Val < 0.05)
significant_gene_results <- dplyr::filter(gene_symbol_results, adj.P.Val < 0.05)
table_dir <- file.path(project_root, "results", "tables")
dir.create(table_dir, recursive = TRUE, showWarnings = FALSE)
write.csv(probe_results, file.path(table_dir, "GSE5406_limma_probe_level_all.csv"), row.names = FALSE)
write.csv(gene_symbol_results, file.path(table_dir, "GSE5406_limma_gene_symbol_selected.csv"), row.names = FALSE)
write.csv(significant_probe_results, file.path(table_dir, "GSE5406_limma_probe_level_significant.csv"), row.names = FALSE)
write.csv(significant_gene_results, file.path(table_dir, "GSE5406_limma_gene_symbol_significant.csv"), row.names = FALSE)
limma_summary <- tibble::tibble(
  study = "GSE5406", n_included = length(included_ids), n_DCM = sum(group == "DCM"),
  n_control = sum(group == "non-failing control"), n_platform_control_probes_removed = sum(control_probe),
  n_features_tested = nrow(expression_test),
  n_mapped_unique_symbol_probes = sum(annotation_map$annotation_status == "mapped"),
  n_ambiguous_probes = sum(annotation_map$annotation_status == "ambiguous"),
  n_unmapped_probes = sum(annotation_map$annotation_status == "unmapped"),
  n_significant_probes = nrow(significant_probe_results),
  n_significant_probes_abs_logFC_ge_1 = sum(abs(significant_probe_results$logFC) >= 1),
  n_unique_symbols_selected = nrow(gene_symbol_results),
  n_significant_symbols = nrow(significant_gene_results),
  annotation_source = "Bioconductor hgu133a.db",
  annotation_version = as.character(utils::packageVersion("hgu133a.db")),
  limma_version = as.character(utils::packageVersion("limma"))
)
write.csv(limma_summary, file.path(table_dir, "GSE5406_limma_summary.csv"), row.names = FALSE)

# Print checks, the top-ranked feature, and selected NPPA/NPPB results for review.
cat("Reference group code: group <- stats::relevel(..., ref = \"non-failing control\")\n")
cat("Platform control probes removed:", sum(control_probe), "\\n")
cat("Features tested:", nrow(expression_test), "\\n")
cat("BH-significant probe features (adjusted p < 0.05):", nrow(significant_probe_results), "\\n")
cat("Significant probe features with |logFC| >= 1:", sum(abs(significant_probe_results$logFC) >= 1), "\\n")
cat("Top-ranked probe, means, and DCM-minus-control logFC:\\n")
print(probe_results[1L, c("probe_id", "gene_symbol", "mean_control", "mean_DCM", "logFC", "P.Value", "adj.P.Val")], n = 1)
cat("Selected NPPA/NPPB gene-symbol results:\\n")
marker_results <- gene_symbol_results |>
  dplyr::filter(gene_symbol %in% c("NPPA", "NPPB"))
if (nrow(marker_results)) {
  print(
    dplyr::select(marker_results, gene_symbol, probe_id, mean_control, mean_DCM, logFC, P.Value, adj.P.Val),
    n = Inf
  )
} else {
  cat("Neither NPPA nor NPPB is represented as a uniquely mapped gene symbol.\\n")
}
cat("Result tables saved in:", table_dir, "\\n")
cat("\\nSession information:\\n")
print(utils::sessionInfo())


'select()' returned 1:many mapping between keys and columns



Reference group code: group <- stats::relevel(..., ref = "non-failing control")


Platform control probes removed: 68 \n

Features tested: 22215 \n

BH-significant probe features (adjusted p < 0.05): 3333 \n

Significant probe features with |logFC| >= 1: 64 \n

... (3 outputs left out)

# A tibble: 2 × 7
  gene_symbol probe_id    mean_control mean_DCM   logFC  P.Value    adj.P.Val
  <chr>       <chr>              <dbl>    <dbl>   <dbl>    <dbl>        <dbl>
1 NPPA        209957_s_at         9.09     11.1  2.00   1.62e-10 0.0000000399
2 NPPB        206801_at          10.1      10.1 -0.0604 8.98e- 1 0.961       


Result tables saved in: /home/user/biom5350/biom5350-dcm-consensus/results/tables \n

\nSession information:\n

R version 4.5.2 (2025-10-31)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 26.04 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/blas/libblas.so.3.12.1 
LAPACK: /usr/lib/x86_64-linux-gnu/lapack/liblapack.so.3.12.1;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
[1] hgu133a.db_3.13.0    org.Hs.eg.db_3.22.0  AnnotationDbi_1.72.0
[4] IRanges_2.44.0       S4Vectors_0.48.1     Biobase_2.70.0      
[7] BiocGenerics_0.56.0  generics_0.1.4       limma_3.66.0        

loaded via a namespace (and not attached):
 

In [8]:

# Q12: Check the top-ranked probe by hand
probe <- "219087_at"

control_mean <- mean(expression_test[probe, group == "non-failing control"])
dcm_mean <- mean(expression_test[probe, group == "DCM"])
difference <- dcm_mean - control_mean

reported_logFC <- probe_results$logFC[probe_results$probe_id == probe]

cat("Control mean:", control_mean, "\n")
cat("DCM mean:", dcm_mean, "\n")
cat("Difference:", difference, "\n")
cat("Reported logFC:", reported_logFC, "\n")
cat("Match:", isTRUE(all.equal(as.numeric(difference),
                                as.numeric(reported_logFC))), "\n")


Control mean: 6.201399 


DCM mean: 8.730875 


Difference: 2.529476 


Reported logFC: 2.529476 


Match: TRUE 


## Stage 3 results: differential expression

The model retained all 102 included samples (86 DCM and 16 non-failing controls). GPL96 platform controls were identified by the `AFFX` probe ID prefix and removed before modeling: 68 control probes removed and 22,215 biological probes tested. No samples were excluded.

The design uses disease status only. The `non-failing control` group is the reference, so the `groupDCM` coefficient and `logFC` are DCM minus control. Limma 3.66.0 empirical Bayes results use BH adjustment; significance is adjusted p < 0.05 with no additional fold-change threshold.

| Result | Count |
|---|---:|
| Probe features tested | 22,215 |
| Probes with one mapped gene symbol | 20,210 |
| Ambiguous probes retained in probe-level results | 836 |
| Unmapped probes retained in probe-level results | 1,169 |
| BH-significant probe features | 3,333 |
| Significant probes with absolute log2FC >= 1 | 64 |
| Gene symbols selected (one probe per symbol) | 12,650 |
| BH-significant selected gene symbols | 2,600 |

Probe annotation used Bioconductor `hgu133a.db` 3.13.0. Its one-to-many mappings were handled by retaining all mapping statuses in the complete probe-level table and selecting only uniquely mapped symbols for the gene-symbol table. The selected probe per symbol is ranked by adjusted p-value, then larger absolute logFC, then lexical probe ID.

The top-ranked feature was probe `219087_at` (ASPN): mean control expression 6.201399, mean DCM expression 8.730875, logFC 2.529476, adjusted p-value 1.101099e-19.

| Gene | Selected probe | Mean control | Mean DCM | logFC | Adjusted p-value |
|---|---|---:|---:|---:|---:|
| NPPA | 209957_s_at | 9.085680 | 11.083675 | 1.997995 | 3.990330e-08 |
| NPPB | 206801_at | 10.119378 | 10.058954 | -0.060424 | 0.960652 |

The analysis code cell above writes the complete probe-level results, selected gene-symbol results, both significant subsets, and a summary CSV under `results/tables/`. It also prints `sessionInfo()` for reproducibility.


In [9]:
before <- read.csv("/tmp/study1_before.csv")
after <- read.csv("../results/tables/GSE5406_limma_probe_level_all.csv")
all.equal(before, after)

[1] TRUE